# Analysis 3 — Step 2: 누수 점검 (naive vs OOF)

- **목적(방법론 핵심)**: OOF를 안 하면 same-video 누수로 R²가 얼마나 부풀려지는지 **수치 시연** → OOF 채택 정당화.
- **입력**: `../Analysis1_title/LR/features_partB_v2.csv` + `step1_channel_features.csv`
- 헤드라인은 **OOF만** 사용 (Background §5-3).

In [1]:
import numpy as np, pandas as pd
import statsmodels.formula.api as smf
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

RS = 42

In [2]:
# 베이스 피처 + 채널 피처 머지
base = pd.read_csv("../Analysis1_title/LR/features_partB_v2.csv", parse_dates=["publish_time"])
chan = pd.read_csv("step1_channel_features.csv")[["video_id", "chan_mean_oof", "chan_mean_naive", "chan_freq"]]
df = base.merge(chan, on="video_id", how="inner")
df["chan_freq_log"] = np.log1p(df["chan_freq"])
hour_labels = ["00-05", "06-11", "12-17", "18-23"]
df["hour_bin"] = pd.Categorical(df["hour_bin"], categories=hour_labels)
print("머지:", df.shape, "| 채널 결측:", int(df["chan_mean_oof"].isna().sum()))

머지: (6249, 22) | 채널 결측: 0


In [3]:
obj = ["title_len", "caps_ratio", "exclaim_cnt", "question_cnt", "has_number", "has_bracket", "tag_cnt"]
ctrl = " + C(hour_bin, Treatment(reference='18-23')) + C(publish_weekday) + C(category)"
F_oof   = "log_views ~ " + " + ".join(obj + ["chan_mean_oof", "chan_freq_log"]) + ctrl
F_naive = "log_views ~ " + " + ".join(obj + ["chan_mean_naive", "chan_freq_log"]) + ctrl

tr, te = train_test_split(df.dropna(subset=["publish_time"]), test_size=0.2, random_state=RS)

def test_r2(F):
    m = smf.ols(F, data=tr).fit()
    t = te[te["category"].isin(set(tr["category"].unique()))]
    return r2_score(t["log_views"], m.predict(t))

In [4]:
r2_oof, r2_naive = test_r2(F_oof), test_r2(F_naive)
print(f"OOF   채널평균 → rand test R² = {r2_oof:.4f}")
print(f"naive 채널평균 → rand test R² = {r2_naive:.4f}  (누수)")
print(f"낙관편향 격차 = {r2_naive - r2_oof:+.4f}  ← OOF 안 하면 이만큼 부풀려짐")

OOF   채널평균 → rand test R² = 0.2956
naive 채널평균 → rand test R² = 0.7366  (누수)
낙관편향 격차 = +0.4410  ← OOF 안 하면 이만큼 부풀려짐


In [5]:
# same-video leakage 차단 수동 검증: 다회 등장 채널 한 곳에서
#   naive 평균 = 자기 포함, OOF = held-out fold라 자기 미포함 → 값이 달라야 정상
ch = df.loc[df["chan_freq"] > 10, "video_id"]
samp = chan[chan["video_id"].isin(ch)].head(3)
print(samp[["video_id", "chan_mean_oof", "chan_mean_naive"]].to_string(index=False))
print("→ OOF ≠ naive (자기 타깃 미포함 확인)")

   video_id  chan_mean_oof  chan_mean_naive
0devsSCkYRY      11.016588        10.881475
7pXcSzCvYvs      10.888779        10.578155
mJ-lFt9_uxI      10.884817        10.881475
→ OOF ≠ naive (자기 타깃 미포함 확인)


## 점검 메모

- naive **0.737** vs OOF **0.296** → 격차 **+0.44**가 same-video 누수 낙관편향.
- 그래서 헤드라인(Step 3)은 OOF만 보고. naive는 "왜 OOF가 필요한가"의 반증으로만 제시.